# Análisis de Datos
## TP N 2
----
### Grupo N° 7
- Aviani, José
- Diaz, José Luis
- Silvera, Ricardo


## Introducción

Para este trabajo elegimos el el dataset Precios Claros – Base SEPA, perteneciente al “Sistema Electrónico de Publicidad de Precios Argentinos (SEPA)" (https://datos.gob.ar/), el cual reúne los precios de comercios minoristas (grandes establecimientos) de más de 70 mil productos en toda la Argentina. Particularmente para este trabajo, seleccionamos el set de datos del establecimiento **Carrefour** ya que era el de mayor tamaño, lo cual es deseable como entrada en un problema de aprendizaje de máquina.
A continuación realizamos el análisis exploratorio de los datos y finalizamos con las conclusiones obtenidas del trabajo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, QuantileTransformer, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.decomposition import PCA
from category_encoders import TargetEncoder
from pandas.api.types import CategoricalDtype

In [ ]:
# Leer el JSON
with open("./dataset/carrefour_dtypes.json", "r") as f:
    info = json.load(f)

dtypes_str = info["dtypes"]
categorical_cols = info["categoricals"]

# Convertir strings de tipo a los tipos correctos
def convertir_dtype(dtype_str):
    if dtype_str.startswith("int"): return "Int64"
    if dtype_str.startswith("float"): return "float"
    if dtype_str == "object": return "string"
    if dtype_str == "bool": return "boolean"
    return "string"

normal_dtypes = {
    col: convertir_dtype(dtype) for col, dtype in dtypes_str.items() if col not in categorical_cols
}

# Leer CSV
carrefour = pd.read_csv("./dataset/carrefour.csv.gz", compression="gzip", dtype=normal_dtypes, sep='|')


# Restaurar categoricas
for col in categorical_cols:
    carrefour[col] = carrefour[col].astype("category")


carrefour.head()

In [ ]:
print(carrefour.describe())

In [ ]:
carrefour.isna().sum()[carrefour.isna().sum() > 0] # columnas con datos faltantes

## 1. Tratamiento de datos faltantes


### 1.1 Eliminación de datos faltantes

Si no hay demasiados valores faltantes o no se los puede imputar de manera confiable, en este caso `productos_precio_unitario_promo2` y `productos_leyenda_promo2` no tienen ningun valor, podemos proceder a borrar las columnas sin afectar al dataset.

El caso de `sucursales_observaciones`, `sucursales_barrio` y `sucursales_numero` no tiene sentido para nuestro modelo, asi que podemos borrarlas.

### 1.2 Imputaciones 

Nos quedan `productos_precio_unitario_promo1`, `productos_leyenda_promo1`. El texto de la promo podemos borrarlo, ya que no tiene mucho sentido en nuestro caso, y para el caso del precio unitario, lo que podemos hacer es asumir que ya que no tiene promo el precio es el precio unitario. Con esta imputación esta columna tiene cierto sentido para el modelo.


In [ ]:

# Condición: la columna 'precio_promocional' es nula
condicion = carrefour['productos_precio_unitario_promo1'].isnull()

# Asignación: en las filas que cumplen la condición, actualiza el valor
carrefour.loc[condicion, 'productos_precio_unitario_promo1'] = carrefour['productos_precio_lista']


## 2. Tratamiento de outliers

### 2.1 Detectamos outliers en base al rango intercuartil

Un dato se considera outlier si es < (Q1 - 1.5 * IQR)) o > (Q3 + 1.5 * IQR)


In [ ]:
# Métodos estadísticos para detectar outliers
datos_a_detectar_outliers = carrefour[['productos_precio_lista', 'productos_precio_unitario_promo1']]

Q1 = datos_a_detectar_outliers.quantile(0.25)
Q3 = datos_a_detectar_outliers.quantile(0.75)

IQR = Q3 - Q1
outliers_iqr = (datos_a_detectar_outliers < (Q1 - 1.5 * IQR)) | (datos_a_detectar_outliers > (Q3 + 1.5 * IQR))

print("Outliers")
print(f"productos_precio_lista: <{Q1['productos_precio_lista'] - 1.5 * IQR['productos_precio_lista']:.02f} o >{Q3['productos_precio_lista'] + 1.5 * IQR['productos_precio_lista']:.02f}")
print(f"productos_precio_unitario_promo1: <{Q1['productos_precio_unitario_promo1'] - 1.5 * IQR['productos_precio_unitario_promo1']:.02f} o >{Q3['productos_precio_unitario_promo1'] + 1.5 * IQR['productos_precio_unitario_promo1']:.02f}")

Como pudimos ver en el trabajo practico anterior la cantidad de outliers grande, para minimizar el impacto de los outliers podemos aplicar logaritmo.

### 2.2 



In [ ]:
# 2. Transformación logarítmica para reducir impacto de outliers
carrefour['productos_precio_lista_log'] = np.log1p(carrefour['productos_precio_lista'])
carrefour['productos_precio_unitario_promo1_log'] = np.log1p(carrefour['productos_precio_unitario_promo1'])

carrefour.describe()[['productos_precio_lista_log','productos_precio_lista', 'productos_precio_unitario_promo1_log', 'productos_precio_unitario_promo1']]

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 6), sharex=True)

# Primer gráfico: datos originales
axes[0].plot(carrefour.index, carrefour['productos_precio_lista'], alpha=0.7, color='gray')
axes[0].set_ylabel('Precio lista (original)')
axes[0].set_title('Precios lista original vs. Precios lista transformación logarítmica')

# Segundo gráfico: datos imputados
axes[1].plot(carrefour.index, carrefour['productos_precio_lista_log'], alpha=0.7, color='salmon')
axes[1].set_ylabel('Precios lista (transformado)')
axes[1].set_xlabel('Número de observación')

plt.tight_layout()
plt.show()
